# Workstreams E and F on Colab (GPU)

Runs the PyTorch parts of the project: the small CNN with the augmentation ablation, Grad-CAM, transfer learning
(MobileNetV2, EfficientNet-B0, ResNet50) and the PlantDoc field-photo test.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Total run time is roughly 1.5–2 hours.
Everything uses the committed `data/splits.csv`, so results line up with the scikit-learn models.

In [ ]:
# 1. Get the code (replace with your team repo URL)
REPO = "https://github.com/<your-team>/tomato-disease-f21dl.git"
!git clone -q $REPO project
%cd project

In [ ]:
# 2. Get the ORIGINAL PlantVillage tomato images (~314 MB) and the PlantDoc tomato field photos (~150 MB)
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /content/pv
!cd /content/pv && git sparse-checkout init --no-cone && echo '/raw/color/Tomato___*/' > .git/info/sparse-checkout && git checkout -q master
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git /content/plantdoc
!cd /content/plantdoc && git sparse-checkout init --no-cone && printf '/train/Tomato*\n/test/Tomato*\n' > .git/info/sparse-checkout && git checkout -q master
!ln -sfn /content/pv/raw/color data/plantvillage_color && ln -sfn /content/plantdoc data/plantdoc
!ls data/plantvillage_color | head -3; find data/plantdoc/ -name '*.jpg' | wc -l

In [ ]:
# 3. Dependencies (torch/torchvision are preinstalled on Colab)
!pip -q install -r requirements.txt
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# 4. Check the shared split is intact: 18133 rows = train 12696 / val 2718 / test 2719
import pandas as pd; s = pd.read_csv("data/splits.csv"); print(len(s)); print(s.split.value_counts())

## Workstream E: small CNN from scratch, augmentation ablation (none / geo / full)

In [ ]:
!python src/cnn.py --aug all --size 128 --epochs 30 --workers 2

In [ ]:
!python src/gradcam.py --model CNN_scratch_full_128 --size 128
!python src/gradcam.py --model CNN_scratch_none_128 --size 128

## Workstream F: transfer learning (frozen backbone, then partial unfreeze)

In [ ]:
!python src/transfer.py --arch all --workers 2

In [ ]:
!python src/gradcam.py --model TL_mobilenet_v2 --size 224

## Results: every model in one table (the same evaluate.py as the classic models)

In [ ]:
!python src/evaluate.py
from IPython.display import Image, display
for f in ["cnn_curves_128.png", "transfer_curves.png", "gradcam_CNN_scratch_full_128_correct.png",
          "gradcam_CNN_scratch_full_128_wrong.png"]:
    display(Image(f"results/figures/{f}", width=800))

In [ ]:
# 5. Save results back to the repo (or download the zip)
!zip -qr results_EF.zip results/tables results/figures
from google.colab import files; files.download("results_EF.zip")